# 11. Custom Workflow Agent — Combine deterministic and agentic steps

Many strong agent applications are not fully autonomous loops. They combine deterministic routing and validation with agentic steps only where open-ended reasoning is valuable.

**Learning goals**
- Route work deterministically before invoking an agentic step.
- Compile a small workflow from explicit stages.
- Keep predictable control flow around flexible model behavior.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# Observability setup — disabled when keys are not present.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")
lf_config = {}

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class WorkflowState(TypedDict):
    question: str
    route: str
    answer: str

## 11.1 Deterministic router

A deterministic router handles cases where the decision rule is known. This keeps the agent from spending model calls on work that code can classify reliably.


In [ ]:
def route(state: WorkflowState) -> dict:
    if "sql" in state["question"].lower():
        return {"route": "database"}
    return {"route": "general"}

In [ ]:
def answer(state: WorkflowState) -> dict:
    text = f"{state['route']} workflow handled: {state['question']}"
    return {"answer": text}

## 11.2 Compile the workflow

A compiled workflow makes the control path reviewable. Each stage has a narrow responsibility, which makes testing and debugging easier.


In [ ]:
builder = StateGraph(WorkflowState)
builder.add_node("route", route)
builder.add_node("answer", answer)
builder.add_edge(START, "route")
builder.add_edge("route", "answer")
builder.add_edge("answer", END)
graph = builder.compile()

In [ ]:
graph.invoke({"question": "Explain the SQL result", "route": "", "answer": ""})

---

## Summary

| Item | Content |
|---|---|
| **Covered** | StateGraph workflows that mix deterministic routing with agentic answer steps |
| **Core idea** | Start from a small deterministic contract before adding model calls or external services. |
| **Next step** | Follow the linked course notebooks and official reference notes listed in this chapter. |

## Reference docs

- [`custom-multi-agent.md`](../../docs/langchain/multi-agent/custom-workflow.md)
- [`workflows-agents.md`](../../docs/langgraph/workflows-agents.md)
- [`runtime.md`](../../docs/langchain/runtime.md)
